# Page 7 — "Frequency Voting": run the chain many times and keep what recurs

Page 6 made one chain longer. This page runs the chain many times and keeps the answer the most
chains agree on. Two axes, measured separately because they answer different questions:

- **the pile** — N chains, each allowed to run to its natural end, and what the vote over them is
  worth against one chain. This is what voting buys.
- **the split** — N chains sharing one fixed budget of `B` tokens, `B/N` each. This is what
  voting costs when the budget is fixed, and it is page 6's closing promise honoured literally.

Both need diversity, so temperature is a measured axis too. At temperature 0 every chain is the
same chain and the vote is a no-op, which the run demonstrates rather than asserts.

Writes `out/page_07/`: `traces.jsonl`, `derived.json`, `manifest.json`. Its `traces.jsonl` is
page 8's input.

The claims contract lives in [`docs/page_7_refactor.md`](../docs/page_7_refactor.md). Shared
plumbing lives in [`harness.py`](harness.py), which ships alongside this notebook.

## 0. How to run this

| Mode | Model | Items | Pile | Temperatures | Hardware | Time |
|---|---|---|---|---|---|---|
| **`REPLAY`** *(default)* | none loaded | 100 | 16 chains | 5 | **no GPU** | **~2 min** |
| `QUICK` | Qwen2.5-3B | 20 | 8 chains | 2 | ~8 GB GPU | ~20 min |
| `FULL` | Qwen2.5-3B | 100 | 16 chains | 5 | 16 GB | ~2 h batched |

**Two things make this affordable, and both are measured rather than assumed.**

*The N axis is one pile, subsampled.* Accuracy at N = 4 is the average over random 4-subsets of
the same 16 chains, so the whole curve costs one pile instead of one sweep per point. Only the
split axis, where the chain length changes with N, has to be generated per point.

*Chains are generated in batches.* N chains of one problem share a prompt, so they are one
`generate` call with `num_return_sequences`. That is the difference between about nineteen hours
and about two. Section 6 prints the estimate for this machine before anything long starts.

**A batch is the unit of reproduction, not a chain.** One seed per group of `GROUP` chains, and
`batch_seed` and `group_index` are recorded on every record, so a group regenerates exactly and a
single chain regenerates only as part of its group. Batching also changes floating-point
reduction order against an unbatched run, so the temperature-0 line is generated unbatched.

**The models are page 6's, not the spec's.** §3 of the spec still names Qwen3, which page 6
retired: nothing blurts unprompted, and Qwen3 does not share the Qwen2.5 tokenizer. Page 7 uses
page 6's checkpoint over page 6's items so the two pages describe one population.

```bash
python -m venv .venv && source .venv/bin/activate
pip install -r requirements.txt
jupyter lab
```

# Part 1 — Setup and the sweep

Configuration, the item set, the chain generator, the generations and the checks on them.
Everything here produces `traces.jsonl` and nothing here draws anything.

**Skip to part 2** if what you came for is how the page's figures were made. In `REPLAY` this
part loads no model and takes about a minute; you are not skipping a wait.

In [ ]:
# ---------------------------------------------------------------------------
# The knobs. Everything else is derived from these.
# ---------------------------------------------------------------------------
RUN_MODE = "REPLAY"       # "REPLAY" (no GPU), "QUICK" (small pile), "FULL"
SEED = 123                # selects items, and seeds every batch of chains
DATASET = "math500_l45"   # page 6's band; "gsm8k" is the fallback section 6 may recommend

PRIMARY = "Qwen/Qwen2.5-3B-Instruct"        # page 6's primary, so the pages compare

PRESETS = {
    # Every figure from the published generations: no GPU, no weights, about two minutes.
    "REPLAY": dict(n_items=100, pool_n=16, temperatures=[0.0, 0.3, 0.6, 0.9, 1.2],
                   diversity_n=8, diversity_items=30, split_n=[1, 4, 16]),
    # A genuine preview: the same model and procedure on a smaller pile. Its numbers are
    # noisier than FULL's, not different in kind.
    "QUICK":  dict(n_items=20, pool_n=8, temperatures=[0.0, 0.9],
                   diversity_n=4, diversity_items=10, split_n=[1, 4]),
    "FULL":   dict(n_items=100, pool_n=16, temperatures=[0.0, 0.3, 0.6, 0.9, 1.2],
                   diversity_n=8, diversity_items=30, split_n=[1, 4, 16]),
}
CFG = PRESETS[RUN_MODE]

# ---- the pile: N chains, each running to its natural end -------------------
# 2048 is page 6's top rung, where 94% of this checkpoint's chains finish on their
# own. It is a safety cap rather than a budget: section 6 prints what a chain at
# this cap actually spends, which is about a third of it.
CHAIN_CAP = 2048
POOL_N = CFG["pool_n"]
# The N axis, built by drawing subsets from the pile rather than by generating.
N_AXIS = [n for n in (1, 2, 4, 8, 16, 32) if n <= POOL_N]
N_RESAMPLES = 64          # random subsets per (item, N)

# ---- the split: N chains sharing one budget --------------------------------
B_TOTAL = 2048            # page 6's top rung again, so N=1 is page 6's right-hand end
SPLIT_N = CFG["split_n"]  # B/N tokens each; page 6 measured the greedy version of this line

# ---- diversity -------------------------------------------------------------
TEMPERATURES = sorted(CFG["temperatures"])
TOP_P = 0.95              # held constant so temperature is the only diversity knob
# The diversity readout is a count of distinct answers, which is far lower variance
# than an accuracy, so it is measured on fewer items and says so wherever it appears.
DIVERSITY_N = CFG["diversity_n"]
DIVERSITY_ITEMS = CFG["diversity_items"]

# ---- generation ------------------------------------------------------------
GROUP = 8                 # chains per generate call, and the unit of reproduction
ANSWER_ALLOWANCE = 32     # page 6's forcing rule, for the chains that do hit their cap
TRACES_URL = None         # where REPLAY reads its generations, once it is hosted

if RUN_MODE == "FULL":
    print("!" * 78)
    print("WARNING  A full reproduction generates roughly two million tokens on one")
    print("         checkpoint. Batched on a 16 GB card that is about two hours; the")
    print("         same run one chain at a time is about nineteen. Section 6 prints")
    print("         the estimate for THIS machine. RUN_MODE = \"REPLAY\" draws every")
    print("         figure from the published traces in about two minutes.")
    print("!" * 78 + "\n")

print(f"mode={RUN_MODE}  dataset={DATASET}  items={CFG['n_items']}")
print(f"the pile   {POOL_N} chains at a {CHAIN_CAP}-token cap  ->  N axis {N_AXIS} "
      f"by subsampling")
print(f"the split  B={B_TOTAL} shared by N in {SPLIT_N}  ->  chain budgets "
      f"{[B_TOTAL // n for n in SPLIT_N]}")
print(f"diversity  T in {TEMPERATURES} at N={DIVERSITY_N} on {DIVERSITY_ITEMS} items")
print(f"top_p={TOP_P}   group={GROUP}   answer allowance={ANSWER_ALLOWANCE}")

mode=REPLAY  dataset=math500_l45  items=100
the pile   16 chains at a 2048-token cap  ->  N axis [1, 2, 4, 8, 16] by subsampling
the split  B=2048 shared by N in [1, 4, 16]  ->  chain budgets [2048, 512, 128]
diversity  T in [0.0, 0.3, 0.6, 0.9, 1.2] at N=8 on 30 items
top_p=0.95   group=8   answer allowance=32


In [2]:
import json
import math
import random
import re
import textwrap
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import requests
import torch

# Shared plumbing, shipped beside this notebook.
import harness

# The inherited grading harness: text-in, text-out, no model dependency.
from reasoning_from_scratch.ch02 import get_device
from reasoning_from_scratch.ch03 import (
    load_math500_test,
    render_prompt,
    extract_final_candidate,
    grade_answer,
)

OUT_DIR = Path("out/page_07")
OUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

TRACES_PATH = OUT_DIR / "traces.jsonl"
DERIVED_PATH = OUT_DIR / "derived.json"
MANIFEST_PATH = OUT_DIR / "manifest.json"
PAGE_06_DIR = Path("out/page_06")

mean, median = harness.mean, harness.median


def read_traces(path):
    """Every record in the log, with the last write per key winning.

    The log is append-only and one key can legitimately appear twice: a configuration
    asked for eight chains and later for sixteen regenerates whatever it had to widen
    the group. Reading without collapsing those counts one chain as two. That does not
    change which answer wins a vote, because doubling preserves order, but it does
    understate `distinct_answers` and `coverage` -- and coverage is the ceiling page 8
    measures itself against, so a silent halving of it would move page 8's whole y-axis.
    """
    latest = {}
    for line in Path(path).open(encoding="utf-8"):
        if not line.strip():
            continue
        r = json.loads(line)
        latest[(r["run_id"], r["config_id"], r["problem_id"], r["sample_index"])] = r
    return list(latest.values())


print("torch", torch.__version__)
print("harness", harness.fingerprint()["version"], harness.fingerprint()["sha256"][:12])

torch 2.10.0+cu128
harness 1.3 7076c1218c63


## 1. The grading harness

Inherited unchanged from ch03, exactly as pages 5 and 6 use it. The next cell adds the shapes
this page generates: a chain cut at its cap and forced to commit, and two chains that write the
same number in different notation, which the vote must count as agreement rather than as two
distinct answers.

In [3]:
CHECKS = [
    # (raw completion, ground truth, expected)
    (r"The answer is \boxed{9}.",                              "9",     True),
    (r"27",                                                     "9",     False),
    (r"\boxed{\frac{1}{2}}",                                   "0.5",   True),   # LaTeX vs decimal
    (r"\boxed{1,024}",                                         "1024",  True),   # thousands separator
    # cut at its cap, then forced to commit:
    ("Common denominator is 12, so 3/4 = 9/12 and 2/3 = 8/12. Now I need to subtr"
     "\n\nFinal answer: " + r"\boxed{\frac{17}{12}}",          r"\frac{17}{12}", True),
    # forced, but it committed to the wrong thing -- forcing must not rescue it.
    ("...\n\nFinal answer: " + r"\boxed{\frac{5}{12}}",        r"\frac{17}{12}", False),
]
rows = [(raw, gt, expected, bool(grade_answer(extract_final_candidate(raw), gt)))
        for raw, gt, expected in CHECKS]
failures = [r for r in rows if r[3] != r[2]]
if failures:
    for raw, gt, expected, got in rows:
        flag = "ok " if got == expected else "!! "
        shown = raw if len(raw) < 58 else raw[:30] + " … " + raw[-25:]
        print(f"{flag}{shown:<60} -> {extract_final_candidate(raw)!r:<22} vs {gt!r:<18} {got}")
    raise AssertionError(f"{len(failures)} grader check(s) behaved unexpectedly")
print(f"grader preflight: {len(rows)}/{len(rows)} as expected")

# The same function is the vote's equivalence relation (section 4), so check it there too:
# two chains that write one number two ways have agreed, and a tally by raw string would
# put them in different buckets and understate the spike this page is about.
print("\nequivalence, which is what the vote will tally by:")
for a, b in ((r"\frac{1}{2}", "0.5"), ("1,024", "1024"), ("17/12", r"\frac{17}{12}"), ("3", "4")):
    print(f"   {a!r:<18} == {b!r:<16} -> {bool(grade_answer(a, b))}")

grader preflight: 6/6 as expected

equivalence, which is what the vote will tally by:
   '\\frac{1}{2}'     == '0.5'            -> True
   '1,024'            == '1024'           -> True
   '17/12'            == '\\frac{17}{12}' -> True
   '3'                == '4'              -> False


## 2. The item set

Page 6's draw, from the same band with the same seed, so the two pages describe one population
and page 6's greedy curve is directly comparable to this page's split axis.

The next cell asserts that against page 6's manifest when that run is on disk, rather than
trusting that two notebooks with the same constants drew the same items.

In [4]:
math500 = load_math500_test(local_path=DATA_DIR / "math500_test.json")

BANDS = {
    "math500_l45": [r for r in math500 if int(r["level"]) in (4, 5)],
    # The fallback section 6's gate may recommend: GSM8K chains run about a third as
    # long, so a given budget splits into chains that can still reason.
    "gsm8k": harness.load_gsm8k(DATA_DIR / "gsm8k_test.jsonl"),
}

band = sorted(BANDS[DATASET], key=lambda r: r["unique_id"])
random.Random(SEED).shuffle(band)
items = band[: CFG["n_items"]]
assert len(items) == CFG["n_items"], \
    f"asked for {CFG['n_items']} items, {DATASET} holds only {len(band)}"

ITEM_SET_HASH = harness.sha256_of(r["unique_id"] for r in items)
print(f"{DATASET}: {len(items)} of {len(band)} available   sha256 {ITEM_SET_HASH[:12]}")

P6_MANIFEST = PAGE_06_DIR / "manifest.json"
SAME_ITEMS_AS_PAGE_6 = None
if P6_MANIFEST.exists():
    p6 = json.loads(P6_MANIFEST.read_text())
    p6_items = p6.get("items", {})
    SAME_ITEMS_AS_PAGE_6 = (p6_items.get("item_set_sha256") == ITEM_SET_HASH)
    print(f"page 6 item set  {str(p6_items.get('item_set_sha256'))[:12]}  "
          f"n={p6_items.get('n')}  seed={p6_items.get('seed')}  dataset={p6_items.get('dataset')}")
    print(f"-> {'identical draw' if SAME_ITEMS_AS_PAGE_6 else 'DIFFERENT draw from page 6'}")
    if not SAME_ITEMS_AS_PAGE_6:
        print("   The two pages then describe different populations and the split axis cannot")
        print("   be compared to page 6's curve. Match n_items, SEED and DATASET.")
else:
    print("out/page_06/manifest.json not found; the page 6 cross-checks below are skipped")

math500_l45: 100 of 262 available   sha256 9ab2b5792042
page 6 item set  9ab2b5792042  n=100  seed=123  dataset=math500_l45
-> identical draw


## 3. What page 6 already measured, and what it predicts

Page 6 swept this checkpoint over these items at five caps. Two things come out of that file and
neither needs a GPU here:

1. **What a chain actually spends**, which is not its cap. That is this notebook's cost model.
2. **How often one chain of `b` tokens is right**, which is the per-chain accuracy every point on
   both axes inherits. On the split axis it predicts the whole line before anything is generated.

**The quantity that decides whether a vote can work is `N x accuracy(B/N)`, the expected number
of correct chains.** Below one, the right answer is usually absent and no tally can find it.

In [5]:
P6_CURVE, P6_SPEND = None, None
P6_TRACES = PAGE_06_DIR / "traces.jsonl"
if P6_TRACES.exists():
    p6_raw = [json.loads(l) for l in P6_TRACES.open(encoding="utf-8") if l.strip()]
    # Page 6's bank holds TWO arms. The cap sweep keys on `budget`; the extension arm
    # keys on `waits` and carries budget=None, and its soft-request inset runs under a
    # pseudo model id. Only the cap arm is a budget curve, so only it is read here --
    # without this filter a None sorts against an int and the cell dies.
    p6_raw = [r for r in p6_raw
              if r.get("model_id") == PRIMARY and r.get("budget") is not None
              and r.get("arm") is None]
    by_b = {}
    for r in p6_raw:
        by_b.setdefault(r["budget"], []).append(r)
    P6_CURVE = {b: mean([r["correct"] for r in rs]) for b, rs in sorted(by_b.items())}
    P6_SPEND = {b: mean([r["tokens_generated"] for r in rs]) for b, rs in sorted(by_b.items())}
    print(f"page 6, {PRIMARY.split('/')[-1]}, greedy, n={len(by_b[max(by_b)])}:\n")
    print(f"  {'cap':>6}{'mean spend':>13}{'of the cap':>12}{'solved':>9}")
    for b in P6_CURVE:
        print(f"  {b:>6}{P6_SPEND[b]:>13,.0f}{P6_SPEND[b] / b:>11.0%}{P6_CURVE[b]:>9.0%}")
    print("\n  A cap is not a spend. At the top of the grid this checkpoint writes about a")
    print("  third of what it is allowed, which is why the pile below is affordable.")
else:
    print("out/page_06/traces.jsonl not found: the cost estimate and the split-axis")
    print("prediction below are skipped, and the run stands on the pilot in section 8 alone.")


def spend_at(cap):
    """What one chain at this cap actually costs, from page 6's measurements. Falls back
    to the cap itself, which is the conservative direction for an estimate."""
    if not P6_SPEND:
        return cap
    at_or_below = [b for b in P6_SPEND if b <= cap]
    return P6_SPEND[max(at_or_below)] if at_or_below else min(cap, min(P6_SPEND.values()))


SPLIT_PREDICTION = []
if P6_CURVE:
    print(f"\nthe split axis, predicted from page 6 before anything is generated:")
    print(f"  {'N':>4}{'tok/chain':>11}{'one chain':>12}{'expected correct chains':>26}")
    for n in sorted(set(SPLIT_N) | {2, 8, 32}):
        per = B_TOTAL // n
        below = [b for b in P6_CURVE if b <= per]
        acc = P6_CURVE[max(below)] if below else None
        exp = None if acc is None else n * acc
        SPLIT_PREDICTION.append(dict(n=n, chain_budget=per, page6_accuracy=acc,
                                     expected_correct=exp, on_axis=n in SPLIT_N))
        shown = (f"{'not measured this short':>26}" if acc is None
                 else f"{acc:>11.0%}{exp:>20.2f}")
        print(f"  {n:>4}{per:>11}{shown}{'' if n in SPLIT_N else '   (not on the axis)'}")
    usable = [p for p in SPLIT_PREDICTION if p["expected_correct"] is not None]
    if usable:
        best = max(usable, key=lambda p: p["expected_correct"])
        one = next(p for p in SPLIT_PREDICTION if p["n"] == 1)
        print(f"\n  expected correct chains peaks at N={best['n']} "
              f"({best['expected_correct']:.2f} of {best['n']})")
        print(f"  and one chain of the whole budget is already right "
              f"{one['page6_accuracy']:.0%} of the time.")
        print("\n  Read that as the warning it is: at a fixed budget on this band, splitting")
        print("  buys a little coverage and pays for it in per-chain accuracy. The split axis")
        print("  is measured anyway, because 'what it costs' is a finding and section 15 is")
        print("  where the run says which way it went.")

page 6, Qwen2.5-3B-Instruct, greedy, n=100:

     cap   mean spend  of the cap   solved
     128          132       103%       7%
     256          258       101%      17%
     512          449        88%      35%
    1024          608        59%      44%
    2048          693        34%      43%

  A cap is not a spend. At the top of the grid this checkpoint writes about a
  third of what it is allowed, which is why the pile below is affordable.

the split axis, predicted from page 6 before anything is generated:
     N  tok/chain   one chain   expected correct chains
     1       2048        43%                0.43
     2       1024        44%                0.88   (not on the axis)
     4        512        35%                1.40
     8        256        17%                1.36   (not on the axis)
    16        128         7%                1.12
    32         64   not measured this short   (not on the axis)

  expected correct chains peaks at N=4 (1.40 of 4)
  and one chain of the 

## 4. When have two chains agreed?

The vote tallies answers, so it needs a rule for when two answers are the same answer. A tally by
raw string would put `\frac{1}{2}` and `0.5` in different buckets and split the spike this page
is about.

**Two chains agree when the inherited grader says they are equivalent.** The same `grade_answer`
that decides correctness decides agreement, so the vote's equivalence relation is not ours to
defend either. Candidates are clustered per problem and the cluster's first-seen form is its
label.

**A chain with no number in it has not voted.** The vote key uses ch03's `number_only` fallback,
so a chain that produced nothing usable lands in a `no answer` class that cannot win. Grading of
an individual chain is unchanged from page 6, which uses the default fallback; section 11 checks
how often those two disagree.

In [6]:
NO_ANSWER = None
NO_ANSWER_LABEL = "(no usable answer)"


def vote_key(text):
    """What a chain contributes to the tally, or None if it contributed nothing."""
    cand = extract_final_candidate(text, fallback="number_only")
    return cand if cand and cand.strip() else NO_ANSWER


_EQ_CACHE = {}


def equivalent(a, b):
    if a == b:
        return True
    k = (a, b) if a <= b else (b, a)
    if k not in _EQ_CACHE:
        try:
            # grade_answer is not documented as symmetric, so ask it both ways.
            _EQ_CACHE[k] = bool(grade_answer(a, b)) or bool(grade_answer(b, a))
        except Exception:
            _EQ_CACHE[k] = False
    return _EQ_CACHE[k]


def cluster(keys):
    """Group answer strings into equivalence classes, first-seen form as the label.
    Returns (label per input, ordered class labels)."""
    labels, order = [], []
    for k in keys:
        if k is NO_ANSWER:
            labels.append(NO_ANSWER)
            continue
        hit = next((lab for lab in order if equivalent(lab, k)), None)
        if hit is None:
            order.append(k)
            hit = k
        labels.append(hit)
    return labels, order


demo_keys = [r"\frac{1}{2}", "0.5", "1,024", "1024", "3", r"\frac{1}{2}", "0.500"]
lab, order = cluster(demo_keys)
print("clustering, on made-up answers so the rule is visible before it is used:")
for k, l in zip(demo_keys, lab):
    print(f"   {k!r:<16} -> class {l!r}")
print(f"   {len(demo_keys)} answers, {len(set(lab))} distinct by string, {len(order)} classes")

clustering, on made-up answers so the rule is visible before it is used:
   '\\frac{1}{2}'   -> class '\\frac{1}{2}'
   '0.5'            -> class '\\frac{1}{2}'
   '1,024'          -> class '1,024'
   '1024'           -> class '1,024'
   '3'              -> class '3'
   '\\frac{1}{2}'   -> class '\\frac{1}{2}'
   '0.500'          -> class '\\frac{1}{2}'
   7 answers, 3 distinct by string, 3 classes


## 5. The tally

Per problem and pile, all of it a reduction of the chains and none of it inference.

| field | what it is |
|---|---|
| `distinct_answers` | equivalence classes among the chains that answered |
| `answer_entropy` | Shannon entropy over the class frequencies, in bits |
| `plurality_answer` | the class with the most votes, `None` on a tie |
| `plurality_correct` | whether that class is the right answer |
| `any_correct` | whether ANY chain got it right, which is the coverage ceiling |
| `no_answer_rate` | chains that produced nothing usable |

**A tie is not a vote.** ch04's `self_consistency_vote` returns no answer when the top frequency
is shared, and this page follows it: a tie grades wrong and is counted separately, because ties
are common at small N and hiding them inside a tie-break would flatter the vote.

In [7]:
def tally(records):
    """One problem, N chains -> the row the figures read."""
    gt = records[0]["ground_truth"]
    labels, _ = cluster([vote_key(r["trace"]) for r in records])

    counts = Counter([l for l in labels if l is not NO_ANSWER])
    n = len(records)
    ranked = counts.most_common()
    top = ranked[0][1] if ranked else 0
    winners = [a for a, c in ranked if c == top]
    tie = len(winners) > 1
    plurality = winners[0] if (winners and not tie) else None
    runner_up = next((c for _, c in ranked[1:]), 0)
    total = sum(counts.values())
    entropy = -sum((c / total) * math.log2(c / total) for c in counts.values()) if total else 0.0

    return dict(
        n_chains=n,
        distinct_answers=len(counts),
        answer_entropy=entropy,
        # 1.0 = every chain said something different, 0.0 = they all agreed
        entropy_normalised=(entropy / math.log2(n)) if n > 1 and total else 0.0,
        plurality_answer=plurality,
        plurality_votes=top,
        plurality_margin=top - runner_up,
        tie=tie,
        plurality_correct=bool(plurality is not None and grade_answer(plurality, gt)),
        any_correct=any(r["correct"] for r in records),
        first_chain_correct=bool(records[0]["correct"]),
        mean_chain_correct=mean([r["correct"] for r in records]),
        no_answer_rate=mean([l is NO_ANSWER for l in labels]),
        truncated_rate=mean([r["truncated"] for r in records]),
        tokens_generated=sum(r["tokens_generated"] for r in records),
        ranked=[(a, c) for a, c in ranked],
    )


def aggregate(rows):
    """Many problems, one configuration -> the point the curves plot."""
    n = len(rows)
    k = sum(r["plurality_correct"] for r in rows)
    lo, hi = harness.wilson(k, n)
    return dict(
        n_items=n,
        plurality_accuracy=k / n, ci_lo=lo, ci_hi=hi,
        coverage=mean([r["any_correct"] for r in rows]),
        single_chain_accuracy=mean([r["mean_chain_correct"] for r in rows]),
        thrown_away=mean([r["any_correct"] and not r["plurality_correct"] for r in rows]),
        mean_distinct=mean([r["distinct_answers"] for r in rows]),
        mean_entropy=mean([r["answer_entropy"] for r in rows]),
        tie_rate=mean([r["tie"] for r in rows]),
        no_answer_rate=mean([r["no_answer_rate"] for r in rows]),
        truncated_rate=mean([r["truncated_rate"] for r in rows]),
        mean_tokens_generated=mean([r["tokens_generated"] for r in rows]),
    )

## Replaying the published run

**`REPLAY` reads generations instead of making them.** Nothing below knows the difference: the
same `samples` structure is built either way, so every figure, check and gate runs identically.

The replay adopts the published `RUN_ID` and asserts the traces cover the item set this notebook
draws. A file that does not match is refused rather than plotted.

In [8]:
samples, model_meta, SWEPT = {}, {}, []


def config_id(temperature, cap):
    """Names a set of generations. Two askings for the same temperature and the same cap
    share their chains rather than repeating them, which is why a configuration is keyed
    by those two and not by how many chains someone wanted."""
    return f"t{temperature:.2f}-c{cap}"


POOL_CONFIG_FOR = lambda t: config_id(t, CHAIN_CAP)

if RUN_MODE == "REPLAY":
    if TRACES_URL and not TRACES_PATH.exists():
        print(f"fetching published generations from {TRACES_URL}")
        TRACES_PATH.write_text(requests.get(TRACES_URL, timeout=300).text, encoding="utf-8")
    assert TRACES_PATH.exists(), (
        f"REPLAY needs published generations at {TRACES_PATH} and found none. Set TRACES_URL "
        f"in the knobs cell, or use RUN_MODE = 'QUICK' to generate your own.")

    raw = read_traces(TRACES_PATH)
    assert raw, f"{TRACES_PATH} is empty"
    RUN_ID = Counter(r["run_id"] for r in raw).most_common(1)[0][0]
    raw = [r for r in raw if r["run_id"] == RUN_ID]

    want = {r["unique_id"] for r in items}
    missing = want - {r["problem_id"] for r in raw}
    assert not missing, (
        f"the published traces miss {len(missing)} of this notebook's {len(want)} problems; "
        f"that file was made with a different seed, dataset or item count.")

    for r in raw:
        if r["problem_id"] in want:
            samples.setdefault(r["model_id"], {}).setdefault(r["config_id"], {}) \
                   .setdefault(r["problem_id"], []).append(r)
    for mid in samples:
        for cid in samples[mid]:
            for pid in samples[mid][cid]:
                samples[mid][cid][pid].sort(key=lambda r: r["sample_index"])

    SWEPT = [PRIMARY] if PRIMARY in samples else sorted(samples)
    assert SWEPT, f"no records for {PRIMARY}; the published traces are a different roster"
    # A replay loads no model, so the precision cannot be read off a live one. It is not
    # unknown either: the run id records it, and pages 5 to 7 all print it on the page.
    # Leaving it null shipped a manifest that recorded no precision while the page
    # asserted bf16, which is the one field a reader would check first.
    quant_from_run = ("bf16" if "bfloat16" in RUN_ID else
                      "8bit" if "8bit" in RUN_ID else
                      "fp16" if "float16" in RUN_ID else None)
    for mid in SWEPT:
        label = mid.split("/")[-1]
        size = re.search(r"(\d+(?:\.\d+)?)B", label)
        model_meta[mid] = dict(id=mid, label=label,
                               params=int(float(size.group(1)) * 1e9) if size else None,
                               quant=quant_from_run, dtype=None, tokenizer_sha256=None,
                               revision=None, replayed=True)
    n = sum(len(v) for m in samples.values() for c in m.values() for v in c.values())
    print(f"replaying {RUN_ID}")
    print(f"  {n:,} generations, {len(want)} items, nothing loaded")
    for cid in sorted(samples[SWEPT[0]]):
        k = sorted({len(v) for v in samples[SWEPT[0]][cid].values()})
        print(f"    {cid:<16} {len(samples[SWEPT[0]][cid]):>3} items x {k} chains")
else:
    print(f"{RUN_MODE}: generations are made below")

AssertionError: REPLAY needs published generations at out/page_07/traces.jsonl and found none. Set TRACES_URL in the knobs cell, or use RUN_MODE = 'QUICK' to generate your own.

In [ ]:
if RUN_MODE != "REPLAY":
    device = get_device()
    DTYPE = harness.pick_dtype(device)
    loaded = harness.load_model(PRIMARY, device=device, dtype=DTYPE, warmup_tokens=16)
    model, tokenizer = loaded["model"], loaded["tokenizer"]
    N_PARAMS, WARMUP_TOK_S = loaded["params"], loaded["tok_per_s"]
    harness.selftest(tokenizer)
    EOS_IDS = harness.end_of_turn_ids(tokenizer)
    SWEPT = [PRIMARY]
    print(f"\nend-of-turn ids: {sorted(EOS_IDS)}")
else:
    device = DTYPE = model = tokenizer = None
    N_PARAMS = WARMUP_TOK_S = EOS_IDS = None
    print("REPLAY: no model loaded, no weights downloaded")

## 6. What this run will cost

Priced from page 6's measured spend rather than from the caps, because the two differ by a factor
of three at the top of the grid, and pricing a cap would put a day on the estimate that the run
does not take.

The batched rate is measured on this machine by generating one real group before the estimate is
printed.

In [ ]:
def plan_cost():
    """(tokens, breakdown) for what the cells below will generate, ignoring anything
    already on disk."""
    n, rows = len(items), []
    rows.append(("the pile, %d chains at cap %d" % (POOL_N, CHAIN_CAP),
                 POOL_N * spend_at(CHAIN_CAP) * n))
    hot = [t for t in TEMPERATURES if t > 0]
    rows.append(("diversity, %d extra temperatures x %d chains x %d items"
                 % (max(len(hot) - 1, 0), DIVERSITY_N, DIVERSITY_ITEMS),
                 max(len(hot) - 1, 0) * DIVERSITY_N * spend_at(CHAIN_CAP) * DIVERSITY_ITEMS))
    rows.append(("the temperature-0 line, one chain per item", spend_at(CHAIN_CAP) * n))
    rows.append(("the split axis, N in %s" % SPLIT_N,
                 sum(k * spend_at(B_TOTAL // k) for k in SPLIT_N if k != 1) * n))
    return sum(v for _, v in rows), rows


TOTAL_TOKENS, COST_ROWS = plan_cost()
for what, v in COST_ROWS:
    print(f"  {what:<52}{v / 1e6:>7.2f} M")
print(f"  {'total':<52}{TOTAL_TOKENS / 1e6:>7.2f} M tokens")

BATCH_TOK_S = None
if RUN_MODE != "REPLAY":
    import time as _time
    _chat = tokenizer.apply_chat_template(
        [{"role": "user", "content": render_prompt(items[0]["problem"])}],
        tokenize=False, add_generation_prompt=True)
    _enc = tokenizer(_chat, return_tensors="pt", add_special_tokens=False).to(model.device)
    _t0 = _time.time()
    with torch.inference_mode():
        _out = model.generate(**_enc, max_new_tokens=64, do_sample=True, temperature=0.9,
                              top_p=TOP_P, top_k=None, num_return_sequences=GROUP,
                              pad_token_id=tokenizer.pad_token_id
                              if tokenizer.pad_token_id is not None else tokenizer.eos_token_id)
    BATCH_TOK_S = (GROUP * 64) / max(_time.time() - _t0, 1e-6)
    print(f"\nmeasured on this machine: {BATCH_TOK_S:,.0f} tok/s at a group of {GROUP}, "
          f"against {WARMUP_TOK_S:,.0f} tok/s one at a time")
    print(f"-> about {TOTAL_TOKENS / BATCH_TOK_S / 3600:.1f} h batched, "
          f"{TOTAL_TOKENS / max(WARMUP_TOK_S, 1e-6) / 3600:.1f} h unbatched")
    print("   Whatever is already in traces.jsonl comes off both figures.")
    del _out

## 7. One group of chains

Page 6's intervention, unchanged, applied to a whole group at once:

1. **Sample** up to `GROUP` chains from one prompt, at `temperature` with `top_p` = 0.95. One
   `generate` call, because the chains of one problem share a prompt.
2. **Force** any chain that hit its cap: append `Final answer: \boxed{` and generate until the
   brace closes. At the pile's cap most chains never need this.
3. **Grade** with the inherited functions.

**How "it stopped on its own" is decided changes under batching, and this is the one place it
would be easy to get silently wrong.** Page 6 could compare the returned length against the cap,
because an unbatched `generate` returns exactly what it produced. A batch returns every row
padded to the longest, so the length carries no information. Here a chain finished on its own if
and only if it emitted an end-of-turn token, and the trace ends there.

In [ ]:
FORCE_STEM = "\n\nFinal answer: \\boxed{"      # the brace we open and it must close


def cut_at_boxed_close(pieces):
    '''We prefilled one open brace with "\\boxed{". Walk the continuation until that
    brace closes, tracking depth so nested LaTeX like \\frac{1}{2} is not cut in half.'''
    depth, kept = 1, []
    for p in pieces:
        out = ""
        for ch in p:
            out += ch
            if ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    kept.append(out)
                    return kept, True
        kept.append(out)
    return kept, False


def group_seed(problem_id, cid, group_index):
    """A stable 63-bit seed per (problem, configuration, group). Recorded on every record
    in the group, so the group regenerates exactly."""
    return int(harness.sha256_of([str(SEED), problem_id, cid, str(group_index)])[:15], 16)


@torch.inference_mode()
def sample_group(question, cap, temperature, k, seed, mdl, tok, allowance=ANSWER_ALLOWANCE):
    """k chains of one problem, in one generate call. Returns k dicts shaped like page 6's."""
    assert k >= 1, f"asked for {k} chains"
    # `num_return_sequences` needs sampling, and transformers is right to refuse it
    # otherwise: greedy decoding from one prompt has one answer, so k copies of it would
    # be one call pretending to be k. Where more than one greedy chain is genuinely
    # wanted -- section 10's check that the replication rule is sound -- they are made
    # one at a time and independently, which is the only thing that makes finding them
    # identical worth anything.
    if temperature == 0.0 and k > 1:
        out = []
        for _ in range(k):
            out.extend(sample_group(question, cap, temperature, 1, seed, mdl, tok, allowance))
        return out

    chat = tok.apply_chat_template([{"role": "user", "content": render_prompt(question)}],
                                   tokenize=False, add_generation_prompt=True)
    prompt_ids = tok(chat, add_special_tokens=False).input_ids
    inp = torch.tensor([prompt_ids], device=mdl.device)
    sampling = temperature > 0
    if sampling:
        torch.manual_seed(seed)
    out = mdl.generate(
        input_ids=inp, attention_mask=torch.ones_like(inp), max_new_tokens=cap,
        # A checkpoint can ship sampling defaults in generation_config, so do_sample alone
        # is not enough: state every decoding parameter here or the manifest records a
        # setup that was never configured.
        do_sample=sampling,
        temperature=temperature if sampling else None,
        top_p=TOP_P if sampling else None, top_k=None,
        num_return_sequences=k,
        pad_token_id=tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id)

    chains = []
    for row in out:
        gen = row[inp.shape[1]:].tolist()
        # The first end-of-turn token is where this chain really ended; everything after
        # it is padding to the longest row in the batch and is not content it wrote.
        stop = next((i for i, t in enumerate(gen) if t in EOS_IDS), None)
        hit_cap = stop is None
        gen = gen[:stop] if stop is not None else gen[:cap]

        own_text = tok.decode(gen, skip_special_tokens=True)
        own_pieces = harness.incremental_token_strings(gen, tok)
        assert "".join(own_pieces) == own_text

        if not hit_cap:
            chains.append(dict(trace=own_text, tokens=own_pieces, own_tokens=len(gen),
                               answer_tokens=0, tokens_generated=len(gen),
                               truncated=False, forced=False, forced_suffix="",
                               answer_closed=None, cut_index=None, answer_start=None))
            continue

        suffix_ids = tok(FORCE_STEM, add_special_tokens=False).input_ids
        suffix_pieces = harness.incremental_token_strings(suffix_ids, tok)
        assert "".join(suffix_pieces) == FORCE_STEM, (
            "the forced suffix does not survive a tokenise/detokenise round trip on this "
            "checkpoint; the page would draw the intervention in the wrong place")
        # The forced answer is greedy at every temperature: it is our intervention rather
        # than the model's chain, and sampling it would put diversity where we injected it.
        forced_inp = torch.tensor([prompt_ids + gen + suffix_ids], device=mdl.device)
        ans = mdl.generate(input_ids=forced_inp,
                           attention_mask=torch.ones_like(forced_inp),
                           max_new_tokens=allowance, do_sample=False,
                           temperature=None, top_p=None, top_k=None,
                           pad_token_id=tok.pad_token_id if tok.pad_token_id is not None
                           else tok.eos_token_id)
        ans_ids = ans[0][forced_inp.shape[1]:].tolist()
        while ans_ids and ans_ids[-1] in EOS_IDS:
            ans_ids.pop()
        kept, closed = cut_at_boxed_close(harness.incremental_token_strings(ans_ids, tok))

        tokens = own_pieces + suffix_pieces + kept
        chains.append(dict(trace="".join(tokens), tokens=tokens, own_tokens=len(gen),
                           answer_tokens=len(kept),
                           tokens_generated=len(gen) + len(kept),   # our suffix is not its cost
                           truncated=True, forced=True, forced_suffix=FORCE_STEM,
                           answer_closed=closed, cut_index=len(own_pieces),
                           answer_start=len(own_pieces) + len(suffix_pieces)))
    return chains

In [ ]:
def render_chain(r, width=88, head=3, tail=2):
    """One chain the way the page draws it."""
    own = "".join(r["tokens"][: r["cut_index"]]) if r["forced"] else r["trace"]
    lines = textwrap.wrap(own.strip().replace("\n", " "), width) or ["(nothing generated)"]
    if len(lines) > head + tail:
        lines = lines[:head] + [f"… [{len(lines) - head - tail} lines elided] …"] + lines[-tail:]
    out = [f"    {ln}" for ln in lines]
    if r["forced"]:
        out[-1] += " ▌"
        out.append(f"    >> Final answer: \\boxed{{{''.join(r['tokens'][r['answer_start']:])}")
    return "\n".join(out)


if RUN_MODE != "REPLAY":
    demo = items[0]
    print(textwrap.fill("Q: " + demo["problem"], 92))
    print("ground truth:", demo["answer"])
    for t in (0.0, TEMPERATURES[-1]):
        cid = config_id(t, CHAIN_CAP)
        got = sample_group(demo["problem"], CHAIN_CAP, t, 3,
                           group_seed(demo["unique_id"], cid, 0), model, tokenizer)
        print(f"\n{'=' * 96}\ntemperature {t}, three chains:")
        for k, g in enumerate(got):
            a = extract_final_candidate(g["trace"])
            print(f"\n  chain {k}   {g['own_tokens']} tok   answer={a!r}   "
                  f"{'CORRECT' if grade_answer(a, demo['answer']) else 'wrong'}")
            print(render_chain(g))
    print("\nAt temperature 0 the three chains are one chain. Above it they are three.")

## 8. The sweep machinery

`ensure_samples` is the only thing that generates. It asks for *k* chains at one temperature and
one cap, and makes only the ones not already on disk, in groups of `GROUP`. Two askings for the
same configuration therefore share it: the diversity family at the best temperature and the pile
are one set of generations, not two.

**At temperature 0 the chains are replicated rather than regenerated**, and generated unbatched.
One chain is made and copied with `replicated_from` recorded, because greedy decoding from an
identical prompt is an identical chain. Section 11 verifies that on the items where more than one
was generated, and anything counting *generations* rather than *records* must filter on
`replicated_from is null`.

In [ ]:
if RUN_MODE != "REPLAY":
    RUN_ID = (f"{RUN_MODE.lower()}-{DATASET}-n{len(items)}-{ITEM_SET_HASH[:8]}"
              f"-cap{CHAIN_CAP}-pool{POOL_N}-p{TOP_P}-{str(DTYPE).rsplit('.', 1)[-1]}"
              f"-{device.type}")
    STORE = harness.TraceStore(TRACES_PATH,
                               RUN_ID, ("model_id", "config_id", "problem_id", "sample_index"))
    STORE.report()

    # Verified on the first few items, where the whole group is generated rather than copied.
    T0_VERIFY_ITEMS = min(5, len(items))


    def ensure_samples(temperature, cap, k, rows, label=""):
        cid = config_id(temperature, cap)
        out, n_gen, n_rep = {}, 0, 0
        replicate = temperature == 0.0

        bar = harness.progress(rows, desc=f"{label or cid:<28}", unit="item")
        for i, row in enumerate(bar):
            pid = row["unique_id"]
            got = [STORE.get(PRIMARY, cid, pid, s) for s in range(k)]
            if all(h is not None for h in got):
                out[pid] = got
                continue

            # Only the groups that are actually missing. Asking a configuration for
            # eight chains and later for sixteen used to regenerate the first eight and
            # append them a second time, which cost the compute twice and left two
            # records under one key in the log.
            step = 1 if replicate else GROUP
            # Greedy: one real chain, copied. Verified in full on the first few items.
            n_real = k if (replicate and i < T0_VERIFY_ITEMS) else (1 if replicate else k)
            for start in range(0, n_real, step):
                size = min(step, n_real - start)
                if all(got[s] is not None for s in range(start, start + size)):
                    continue                          # this group is already on disk
                g_index = start // step
                seed = group_seed(pid, cid, g_index)
                made = sample_group(row["problem"], cap, temperature, size, seed,
                                    model, tokenizer)
                n_gen += size
                for j, g in enumerate(made):
                    s = start + j
                    extracted = extract_final_candidate(g["trace"])
                    got[s] = STORE.append({
                        "model_id": PRIMARY, "config_id": cid,
                        "problem_id": pid, "sample_index": s,
                        "group_index": g_index, "batch_seed": seed, "batch_size": size,
                        "level": int(row["level"]), "subject": row["subject"],
                        "question": row["problem"], "ground_truth": row["answer"],
                        "temperature": temperature,
                        "top_p": TOP_P if temperature > 0 else None,
                        "chain_budget": cap, "budget_total": B_TOTAL,
                        "trace": g["trace"], "tokens": g["tokens"],
                        "cut_index": g["cut_index"], "answer_start": g["answer_start"],
                        "truncated": g["truncated"], "forced": g["forced"],
                        "forced_suffix": g["forced_suffix"],
                        "answer_closed": g["answer_closed"],
                        "extracted_answer": extracted,
                        "vote_key": vote_key(g["trace"]),
                        "correct": bool(grade_answer(extracted, row["answer"])),
                        "own_tokens": g["own_tokens"], "answer_tokens": g["answer_tokens"],
                        "tokens_generated": g["tokens_generated"],
                        "replicated_from": None,
                    })
            # Greedy only: the slots past the one real chain are copies of it.
            for s in range(n_real, k):
                if got[s] is None:
                    got[s] = STORE.append(dict(got[0], sample_index=s, batch_seed=None,
                                               replicated_from=0))
                    n_rep += 1
            assert all(g is not None for g in got), \
                f"{cid} {pid}: {sum(g is None for g in got)} of {k} chains never filled"
            out[pid] = got
            bar.set_postfix(gen=n_gen, repl=n_rep)

        samples.setdefault(PRIMARY, {}).setdefault(cid, {}).update(out)
        return out
else:
    def ensure_samples(*a, **k):
        raise RuntimeError("REPLAY does not generate; the published traces are the run")

## 9. The pile, the diversity family, and the split

Order matters and it is not arbitrary. The diversity family runs first on a small item set,
because it is what chooses the temperature the pile is generated at. Then the pile, on every
item, which is figure 1, figure 2's main line, the handoff number and page 8's candidates. Then
the split axis, which is the only thing here that generates a point per N.

In [ ]:
if RUN_MODE != "REPLAY":
    for t in [x for x in TEMPERATURES if x > 0]:
        ensure_samples(t, CHAIN_CAP, DIVERSITY_N, items[:DIVERSITY_ITEMS],
                       label=f"diversity T={t}")
    ensure_samples(0.0, CHAIN_CAP, DIVERSITY_N, items[:DIVERSITY_ITEMS],
                   label="diversity T=0.0")
    print("\ndiversity family complete")

PRIMARY_SAMPLES = samples[SWEPT[0]]


def rows_at(cid, k=None, only=None):
    """The per-problem tallies at one configuration. `k` trims each vote to its first k
    chains; `only` restricts to a set of problem ids."""
    out = []
    for p in items:
        if only is not None and p["unique_id"] not in only:
            continue
        rs = PRIMARY_SAMPLES.get(cid, {}).get(p["unique_id"])
        if rs and (k is None or len(rs) >= k):
            out.append(tally(rs[:k] if k else rs))
    return out


# The pile's temperature is read off the diversity family rather than chosen: the one
# whose vote solves the most problems, ties broken toward the lower temperature.
DIV_IDS = {p["unique_id"] for p in items[:DIVERSITY_ITEMS]}
D_AGG = {}
for t in TEMPERATURES:
    rs = rows_at(config_id(t, CHAIN_CAP), DIVERSITY_N, only=DIV_IDS)
    if rs:
        D_AGG[t] = aggregate(rs)
assert D_AGG, "no diversity family on disk; nothing can choose the pile's temperature"
T_BEST = max(D_AGG, key=lambda t: (D_AGG[t]["plurality_accuracy"], -t))

print(f"diversity family: N={DIVERSITY_N} chains on {len(DIV_IDS)} items")
print(f"  {'T':>5}{'distinct':>10}{'entropy':>9}{'one chain':>11}{'vote':>7}{'coverage':>10}")
for t, a in sorted(D_AGG.items()):
    print(f"  {t:>5}{a['mean_distinct']:>10.1f}{a['mean_entropy']:>9.2f}"
          f"{a['single_chain_accuracy']:>11.0%}{a['plurality_accuracy']:>7.0%}"
          f"{a['coverage']:>10.0%}")
print(f"\n-> the pile is generated at T={T_BEST}")
print(f"   Read the distinct-answer column, not the accuracies: n={len(DIV_IDS)} resolves")
print(f"   a count of answers and does not resolve a few points of accuracy.")

In [ ]:
if RUN_MODE != "REPLAY":
    ensure_samples(T_BEST, CHAIN_CAP, POOL_N, items, label=f"the pile T={T_BEST}")
    ensure_samples(0.0, CHAIN_CAP, POOL_N, items, label="the pile T=0")
    print("\npile complete")

POOL_CONFIG = config_id(T_BEST, CHAIN_CAP)
POOL_N = min((len(v) for v in PRIMARY_SAMPLES.get(POOL_CONFIG, {}).values()), default=0)
assert POOL_N >= 2, (
    f"the pile at {POOL_CONFIG} holds {POOL_N} chains per problem, which is not a pile.")
N_AXIS = [n for n in N_AXIS if n <= POOL_N] or [POOL_N]
if N_AXIS[-1] != POOL_N:
    N_AXIS.append(POOL_N)
print(f"pile {POOL_CONFIG}: {POOL_N} chains x "
      f"{len(PRIMARY_SAMPLES[POOL_CONFIG])} problems, N axis {N_AXIS}")

In [ ]:
if RUN_MODE != "REPLAY":
    for n in SPLIT_N:
        if n == 1:
            continue                      # N=1 of the split axis IS the pile's first chain
        ensure_samples(T_BEST, B_TOTAL // n, n, items, label=f"split N={n}")
    print("\nsplit axis complete")

## 10. Sweep integrity

Six checks. Three are specific to this page:

- **Check 3** is that the split axis really shares one budget, and that the forcing allowance
  charged on top of it is reported rather than buried.
- **Check 4** is that sampling sampled. Above temperature 0 the chains of one pile must not be
  identical, and at temperature 0 they must be.
- **Check 5** is that the vote key never loses a right answer the grader found.

In [ ]:
all_records = [r for cid in PRIMARY_SAMPLES for rs in PRIMARY_SAMPLES[cid].values() for r in rs]
generated = [r for r in all_records if r.get("replicated_from") is None]
print(f"{len(all_records):,} records  ({len(generated):,} generations, "
      f"{len(all_records) - len(generated):,} replicated at temperature 0)\n")

bad = [r for r in all_records if "".join(r["tokens"]) != r["trace"]]
print(f"1. traces reconstruct from tokens .......... {len(bad)} violations")
assert not bad, "a trace's token strings do not join back to its text"

forced = [r for r in all_records if r["forced"]]
unclosed = [r for r in forced if r["answer_closed"] is False]
unclosed_rate = len(unclosed) / max(len(forced), 1)
print(f"2. forced answers closed the brace ......... {1 - unclosed_rate:.1%} "
      f"({len(unclosed)} of {len(forced)} ran out of allowance)")
print(f"   {len(forced):,} of {len(all_records):,} chains hit their cap at all; at the pile's "
      f"cap of {CHAIN_CAP}\n   most chains finish on their own and are never forced.")

print("\n3. what each configuration spent per problem:")
print(f"   {'config':<16}{'chains':>7}{'thinking':>10}{'forced':>8}{'total':>9}{'of B':>7}")
BUDGET_AUDIT = []
for cid in sorted(PRIMARY_SAMPLES):
    by_item = PRIMARY_SAMPLES[cid]
    cap = next(iter(by_item.values()))[0]["chain_budget"]
    is_split = cap < CHAIN_CAP
    k = max(1, B_TOTAL // cap) if is_split else median([len(v) for v in by_item.values()])
    think = mean([sum(r["own_tokens"] for r in rs[:k]) for rs in by_item.values()])
    ans = mean([sum(r["answer_tokens"] for r in rs[:k]) for rs in by_item.values()])
    BUDGET_AUDIT.append(dict(config_id=cid, axis="split" if is_split else "pile",
                             chains=k, mean_thinking=think, mean_forced=ans,
                             mean_total=think + ans, of_budget=(think + ans) / B_TOTAL))
    print(f"   {cid:<16}{k:>7}{think:>10,.0f}{ans:>8,.0f}{think + ans:>9,.0f}"
          f"{(think + ans) / B_TOTAL:>7.2f}x   {'split' if is_split else 'pile'}")
over = [a for a in BUDGET_AUDIT if a["axis"] == "split" and a["mean_thinking"] > B_TOTAL + 1]
assert not over, (f"{[a['config_id'] for a in over]} thought for longer than B; the split axis "
                  f"does not mean what it says")
print("   The split axis is matched in thinking tokens. The forced answer is overhead we")
print("   added and it grows with N, because every chain pays it once. The pile is not")
print("   budget matched and is not meant to be: it is what N chains cost.")

In [ ]:
print("4. what sampling did, over generated chains only:")
DIVERSITY_CHECK, T0_IDENTICAL = [], None
for cid in sorted(PRIMARY_SAMPLES):
    real = {pid: [r for r in rs if r.get("replicated_from") is None]
            for pid, rs in PRIMARY_SAMPLES[cid].items()}
    real = {pid: rs for pid, rs in real.items() if len(rs) > 1}
    if not real:
        print(f"   {cid:<16} one generated chain per item; nothing to compare")
        continue
    t = next(iter(real.values()))[0]["temperature"]
    ident = mean([len({r["trace"] for r in rs}) == 1 for rs in real.values()])
    DIVERSITY_CHECK.append(dict(config_id=cid, temperature=t, n_items_compared=len(real),
                                all_identical_rate=ident))
    print(f"   {cid:<16} T={t:<5} {len(real):>3} items compared, all chains identical on "
          f"{ident:>4.0%}")
    if t > 0:
        assert ident < 1.0, (
            f"{cid} sampled at T={t} and produced identical chains on every item; do_sample "
            f"did not take effect and the diversity axis is measuring nothing")
    else:
        T0_IDENTICAL = ident if T0_IDENTICAL is None else min(T0_IDENTICAL, ident)
        assert ident == 1.0, (
            "greedy decoding produced different chains from the same prompt, so the "
            "replication rule in section 8 is unsound and every temperature-0 vote built "
            "on it is a fabrication. Check that do_sample really is False.")

n_replicated = sum(1 for r in all_records if r.get("replicated_from") is not None)
if T0_IDENTICAL is None and n_replicated:
    print(f"\n   !! {n_replicated:,} records are temperature-0 replicas and NOTHING here "
          f"verified the\n      rule that licenses them. Raise T0_VERIFY_ITEMS and re-run, or "
          f"treat every\n      T=0 vote as unchecked.")

lost = [r for r in all_records if r["correct"] and vote_key(r["trace"]) is NO_ANSWER]
print(f"\n5. right answers the tally could not count .. {len(lost)} of "
      f"{sum(r['correct'] for r in all_records)} correct chains")

seeds = [(r["problem_id"], r["config_id"], r.get("group_index"), r.get("batch_seed"))
         for r in generated if r["temperature"] > 0]
uniq = {(p, c, g): s for p, c, g, s in seeds}
print(f"6. one seed per group, and groups distinct .. {len(set(uniq.values()))} seeds "
      f"for {len(uniq)} groups")
assert len(set(uniq.values())) == len(uniq) or not uniq, "two groups share a seed"

INTEGRITY = dict(n_records=len(all_records), n_generated=len(generated),
                 unclosed_forced_rate=unclosed_rate, forced_rate=len(forced) / len(all_records),
                 budget_audit=BUDGET_AUDIT, diversity_check=DIVERSITY_CHECK,
                 n_uncountable_correct=len(lost), t0_identical_rate=T0_IDENTICAL)

# Part 2 — What the page shows

Every figure, number and gate the page renders, derived from what part 1 produced and written to
`out/page_07/`. No generation happens here.

## 11. Figure 1 — the pile, with a real tail

One problem, `POOL_N` chains, tallied into a ranked histogram of distinct answers.

The specimen is **curated from the pile, not generated for the figure**: the vote must land on
the right answer and the tail must be a tail, so the figure shows the mechanism the page claims
rather than a lucky unanimous case.

**Do not bucket the singletons.** The row of ones is the argument.

In [ ]:
FIG1_ROWS = {p["unique_id"]: tally(PRIMARY_SAMPLES[POOL_CONFIG][p["unique_id"]][:POOL_N])
             for p in items if p["unique_id"] in PRIMARY_SAMPLES.get(POOL_CONFIG, {})}
assert FIG1_ROWS, f"no chains at {POOL_CONFIG}"

candidates = []
for pid, row in FIG1_ROWS.items():
    if not row["plurality_correct"] or row["distinct_answers"] < 3:
        continue                     # want a spike, and a tail to put beside it
    singletons = sum(1 for _, c in row["ranked"] if c == 1)
    candidates.append((-singletons, -row["plurality_margin"], pid))
candidates.sort()

SPECIMEN_IS_FALLBACK = False
if candidates:
    SPECIMEN_ID = candidates[0][2]
else:
    SPECIMEN_IS_FALLBACK = True
    SPECIMEN_ID = max(FIG1_ROWS, key=lambda p: FIG1_ROWS[p]["distinct_answers"])
    print("WARNING: no problem has a correct plurality with a tail of at least three distinct\n"
          "         answers. Figure 1 has no honest specimen; derived.json flags it and it must\n"
          "         not be translated. Figure 2 is unaffected.\n")

spec = FIG1_ROWS[SPECIMEN_ID]
spec_chains = PRIMARY_SAMPLES[POOL_CONFIG][SPECIMEN_ID][:POOL_N]
print(textwrap.fill("Q: " + spec_chains[0]["question"], 92))
print("ground truth:", spec_chains[0]["ground_truth"])
print(f"\n{spec['n_chains']} chains at T={T_BEST}, {spec['distinct_answers']} distinct answers\n")
print(f"  {'answer':<26}{'votes':>6}   tally")
for ans, c in spec["ranked"]:
    mark = "  <- correct" if grade_answer(ans, spec_chains[0]["ground_truth"]) else ""
    print(f"  {str(ans)[:25]:<26}{c:>6}   {'#' * c}{mark}")
if spec["no_answer_rate"]:
    print(f"  {NO_ANSWER_LABEL:<26}{round(spec['no_answer_rate'] * spec['n_chains']):>6}")
print(f"\nvote -> {spec['plurality_answer']!r}   "
      f"{'CORRECT' if spec['plurality_correct'] else 'wrong'}   margin {spec['plurality_margin']}")

# The shape of the whole population, which is the claim the specimen illustrates.
SHAPE = dict(
    mean_distinct=mean([r["distinct_answers"] for r in FIG1_ROWS.values()]),
    singleton_share=mean([sum(1 for _, c in r["ranked"] if c == 1) / max(len(r["ranked"]), 1)
                          for r in FIG1_ROWS.values()]),
    mean_top_votes=mean([r["plurality_votes"] for r in FIG1_ROWS.values()]),
    n_chains=POOL_N)
print(f"\nacross all {len(FIG1_ROWS)} problems: {SHAPE['mean_distinct']:.1f} distinct answers "
      f"per {POOL_N} chains,\n{SHAPE['singleton_share']:.0%} of answer classes appear exactly "
      f"once, and the top class holds {SHAPE['mean_top_votes']:.1f} votes.")

In [ ]:
# Two chains that agreed on a WRONG answer usually made different mistakes, and that is
# the payoff for opening a bar.
labels, _ = cluster([vote_key(r["trace"]) for r in spec_chains])
groups = {}
for lab, r in zip(labels, spec_chains):
    groups.setdefault(NO_ANSWER_LABEL if lab is NO_ANSWER else lab, []).append(r)

for ans, rs in sorted(groups.items(), key=lambda kv: -len(kv[1]))[:3]:
    ok = ans != NO_ANSWER_LABEL and bool(grade_answer(ans, spec_chains[0]["ground_truth"]))
    print(f"\n{'=' * 96}\nanswer {ans!r}  x{len(rs)}  {'CORRECT' if ok else 'wrong'}")
    for r in rs[:2]:
        print(f"\n  chain {r['sample_index']}   {r['own_tokens']} tok   "
              f"truncated={r['truncated']}")
        print(render_chain(r))

In [ ]:
import matplotlib.pyplot as plt

ranked = spec["ranked"]
counts = [c for _, c in ranked]
is_right = [bool(grade_answer(a, spec_chains[0]["ground_truth"])) for a, _ in ranked]

fig, ax = plt.subplots(figsize=(7.4, max(2.6, 0.32 * len(ranked) + 1.4)))
ax.barh(range(len(ranked)), counts,
        color=["#3b4a6b" if r else "#c4c0b8" for r in is_right], height=0.66)
ax.set_yticks(range(len(ranked)))
ax.set_yticklabels([str(a)[:18] for a, _ in ranked], fontsize=8)
ax.invert_yaxis()
ax.set_xlabel("chains landing on this answer")
ax.set_xticks(range(0, max(counts) + 1))
ax.spines[["top", "right"]].set_visible(False)
ax.set_title(f"{spec['n_chains']} chains, T={T_BEST}   ·   {spec['distinct_answers']} distinct "
             f"answers, {sum(1 for c in counts if c == 1)} of them once",
             fontsize=9, loc="left", color="#555")
fig.tight_layout()
fig.savefig(OUT_DIR / "figure_1_tally.png", dpi=160)
plt.show()

## 12. Figure 2 — what the vote buys

Accuracy against N, where a point at N is the average over `N_RESAMPLES` random subsets of the
same pile. The chain length is held fixed, so the only thing changing along the axis is **how
many candidates the vote may look at**.

**The temperature-0 line is flat by construction**, because N copies of one greedy chain is one
greedy chain, and that is the page's cleanest demonstration that the vote needs diversity.

The gain is tested rather than described: an exact McNemar over the items where one chain and the
vote disagree.

In [ ]:
def subsets_for(pid, n, repeats, size=None):
    """The draws a point averages over. Exhaustive when the configuration holds exactly
    n chains, because then there is only one subset to draw."""
    size = POOL_N if size is None else size
    if n >= size:
        return [list(range(size))]
    rng = random.Random(harness.sha256_of([str(SEED), pid, str(n), str(size)])[:15])
    return [rng.sample(range(size), n) for _ in range(repeats)]


def curve_point(cid, n):
    """One point on either axis, computed the same way on both.

    The mean over random n-subsets of whatever chains the configuration holds. Where a
    configuration holds exactly n, `subsets_for` returns the single full subset and this
    degenerates to one tally, so the two axes cannot disagree at a shared configuration
    for a reason that is ours rather than the model's. That mattered: the split axis
    used to read its N=1 point off chain 0 alone while the pile averaged over all
    sixteen single draws, and the same configuration appeared on the page twice with
    five points between the two numbers.
    """
    per_item = []
    for p in items:
        rs = PRIMARY_SAMPLES.get(cid, {}).get(p["unique_id"])
        if not rs or len(rs) < n:
            continue
        ts = [tally([rs[i] for i in idx])
              for idx in subsets_for(p["unique_id"], n, N_RESAMPLES, len(rs))]
        per_item.append({k: mean([t[k] for t in ts]) for k in
                         ("plurality_correct", "any_correct", "tie", "mean_chain_correct",
                          "distinct_answers", "tokens_generated", "truncated_rate")})
    if not per_item:
        return None
    m = lambda k: mean([x[k] for x in per_item])
    return dict(n=n, n_items=len(per_item),
                accuracy=m("plurality_correct"), coverage=m("any_correct"),
                tie_rate=m("tie"), single_chain_accuracy=m("mean_chain_correct"),
                mean_distinct=m("distinct_answers"),
                # What N chains actually COST. A chain stops long before its cap, so this
                # is the real spend rather than n times the budget.
                mean_tokens=m("tokens_generated"),
                truncated_rate=m("truncated_rate"),
                n_draws=len(per_item) * N_RESAMPLES)


def pile_curve(cid):
    return [pt for n in N_AXIS if (pt := curve_point(cid, n))]


PILE = {}
for t in sorted({T_BEST, 0.0}):
    cid = config_id(t, CHAIN_CAP)
    if cid in PRIMARY_SAMPLES:
        PILE[t] = pile_curve(cid)

for t, pts in PILE.items():
    print(f"\nT={t}, {POOL_N} chains at a {CHAIN_CAP}-token cap, n={len(items)} items")
    print(f"  {'N':>4}{'vote':>8}{'coverage':>11}{'ties':>7}{'draws':>9}")
    for p in pts:
        print(f"  {p['n']:>4}{p['accuracy']:>8.1%}{p['coverage']:>11.1%}"
              f"{p['tie_rate']:>7.0%}{p['n_draws']:>9,}")

# The headline contrast, paired and tested: one chain against the whole pile.
hot = PILE[T_BEST]
one = {p["unique_id"]: PRIMARY_SAMPLES[POOL_CONFIG][p["unique_id"]][0]["correct"]
       for p in items if p["unique_id"] in PRIMARY_SAMPLES[POOL_CONFIG]}
full = {pid: tally(PRIMARY_SAMPLES[POOL_CONFIG][pid][:POOL_N])["plurality_correct"]
        for pid in one}
fixed, broken, p_value = harness.mcnemar(full, one)
GAIN_PTS = (mean(list(full.values())) - mean(list(one.values()))) * 100
VOTE_BEATS_ONE_CHAIN = GAIN_PTS > 0 and p_value < 0.05

print(f"\none sampled chain      {mean(list(one.values())):.0%}")
print(f"the vote over {POOL_N:<2}     {mean(list(full.values())):.0%}")
print(f"the gain               {GAIN_PTS:+.0f} points   ({fixed} fixed, {broken} broken, "
      f"exact McNemar p={p_value:.4f})")

# The contrast the PAGE makes, which is not the one above. Bare chain of thought is one
# GREEDY chain of the same length on the same items -- page 5's `bare` condition and page
# 6's top rung. The contrast above is against one SAMPLED chain, which is the
# self-consistency literature's baseline and a different question: sampling at T* is not
# what a reader who runs the model once gets. Both are reported, and this is the blocking
# one, because "voting improves on chain of thought" is the sentence the page wants.
GREEDY_CONFIG = config_id(0.0, CHAIN_CAP)
BARE, BARE_GAIN_PTS, VOTE_BEATS_BARE_COT = None, None, False
if GREEDY_CONFIG in PRIMARY_SAMPLES:
    bare = {pid: PRIMARY_SAMPLES[GREEDY_CONFIG][pid][0]["correct"]
            for pid in full if pid in PRIMARY_SAMPLES[GREEDY_CONFIG]}
    paired = {pid: full[pid] for pid in bare}
    b_fixed, b_broken, b_p = harness.mcnemar(paired, bare)
    BARE_GAIN_PTS = (mean(list(paired.values())) - mean(list(bare.values()))) * 100
    VOTE_BEATS_BARE_COT = BARE_GAIN_PTS > 0 and b_p < 0.05
    BARE = dict(bare_cot=mean(list(bare.values())), vote=mean(list(paired.values())),
                gain_pts=BARE_GAIN_PTS, p=b_p, fixed=b_fixed, broken=b_broken,
                n_items=len(bare), config_id=GREEDY_CONFIG)
    print(f"\nbare chain of thought  {BARE['bare_cot']:.0%}   (one greedy chain, the same "
          f"{CHAIN_CAP}-token cap)")
    print(f"the vote over {POOL_N:<2}     {BARE['vote']:.0%}")
    print(f"the gain               {BARE_GAIN_PTS:+.0f} points   ({b_fixed} fixed, "
          f"{b_broken} broken, exact McNemar p={b_p:.4f})")
else:
    print(f"\nno greedy pile at {GREEDY_CONFIG}: the contrast against bare chain of thought "
          f"cannot be made,\nand it is the one the page rests on.")

for label, ok, n in (("one sampled chain", VOTE_BEATS_ONE_CHAIN, len(one)),
                     ("bare chain of thought", VOTE_BEATS_BARE_COT,
                      BARE["n_items"] if BARE else 0)):
    if not ok:
        print(f"\n!! The vote does not beat {label} at p<0.05. That is either a real null or an")
        print(f"   underpowered one: at n={n} a paired difference under about eight points will")
        print("   not resolve. Options are a larger pile, more items, or saying plainly that the")
        print("   gain is bounded rather than shown. Do not describe an unresolved gain as one.")

## 13. The same figure, at a fixed total budget

The other axis, and page 6's closing promise honoured literally: instead of N chains of whatever
length they want, N chains sharing `B` tokens between them.

Page 6 measured the greedy version of this line already, so its points are overlaid from that
run rather than regenerated. What is new here is the sampled version, which is the fair
comparison: at a fixed budget, does sampling and voting beat spending the lot on one chain?

In [ ]:
SPLIT = []
for n in SPLIT_N:
    cid = POOL_CONFIG if n == 1 else config_id(T_BEST, B_TOTAL // n)
    pt = curve_point(cid, n)          # the same estimator the pile uses
    if pt is None:
        continue
    SPLIT.append(dict(chain_budget=B_TOTAL // n, temperature=T_BEST,
                      page_06_greedy=(P6_CURVE or {}).get(B_TOTAL // n),
                      plurality_accuracy=pt["accuracy"], **pt))

print(f"one budget of {B_TOTAL} tokens, split N ways, T={T_BEST}, n={len(items)}")
print(f"  {'N':>4}{'cap each':>10}{'spent':>9}{'the vote':>10}{'coverage':>10}"
      f"{'cut off':>9}{'page 6':>9}")
for s in SPLIT:
    g = "-" if s["page_06_greedy"] is None else f"{s['page_06_greedy']:.0%}"
    print(f"  {s['n']:>4}{s['chain_budget']:>10}{s['mean_tokens']:>9,.0f}"
          f"{s['plurality_accuracy']:>10.0%}{s['coverage']:>10.0%}"
          f"{s['truncated_rate']:>9.0%}{g:>9}")

peak = max(SPLIT, key=lambda s: s["plurality_accuracy"])
SPLIT_PAYS = peak["n"] > 1
print(f"\nthe best split is N={peak['n']} ({peak['plurality_accuracy']:.0%})")

# Read the `cut off` column before concluding anything about voting. Dividing a fixed
# budget does two things at once -- it adds chains AND it shortens them -- and on this
# band the second dominates completely. Compare a split point against the PILE point
# with the same number of chains, which spends more but lets each chain finish: that
# comparison isolates the cap from the split, and it is the one the page rests on.
print("\nthe same N, capped against uncapped:")
print(f"  {'N':>4}{'capped':>9}{'spent':>9}{'cut off':>9}   |{'uncapped':>10}{'spent':>9}")
for sp in SPLIT:
    up = next((p for p in (PILE.get(T_BEST) or []) if p["n"] == sp["n"]), None)
    if up is None:
        continue
    print(f"  {sp['n']:>4}{sp['plurality_accuracy']:>9.0%}{sp['mean_tokens']:>9,.0f}"
          f"{sp['truncated_rate']:>9.0%}   |{up['accuracy']:>10.0%}{up['mean_tokens']:>9,.0f}")

if not SPLIT_PAYS:
    print("\n-> At a fixed budget, on this band, the best thing to do with the budget is NOT")
    print("   to split it. But read that as a statement about the CAP rather than about the")
    print("   vote: at N=16 every chain is cut off mid-thought, and page 6 measured a chain")
    print("   that short at 7% on its own. The vote is being asked to pool fragments.")
    print("   Splitting is not what fails here; capping is.")
else:
    print(f"\n-> Splitting pays at this budget: {peak['n']} chains of {peak['chain_budget']} "
          f"tokens beat one of {B_TOTAL}.")

In [ ]:
fig, (ax, axs) = plt.subplots(1, 2, figsize=(11.2, 4.3),
                              gridspec_kw=dict(width_ratios=[1.35, 1], wspace=0.28))
INK = {0.0: "#9aa0ab"}
for t in sorted(PILE):
    pts = PILE[t]
    ax.plot([p["n"] for p in pts], [p["accuracy"] * 100 for p in pts], marker="o", ms=4.5, lw=2,
            color=INK.get(t, "#3b4a6b"),
            label=f"the vote, T={t}" + ("  (one voter, N times)" if t == 0 else ""))
ax.plot([p["n"] for p in hot], [p["coverage"] * 100 for p in hot], ls="--", lw=1.2,
        color="#a2704a", label="the right answer is somewhere in the pile")
ax.set_xscale("log", base=2)
ax.set_xticks(N_AXIS)
ax.set_xticklabels([str(n) for n in N_AXIS])
ax.set_xlabel(f"chains, each free to run to its own end (cap {CHAIN_CAP})")
ax.set_ylabel("share of problems solved (%)")
ax.set_ylim(0, min(100, max(p["coverage"] for p in hot) * 100 + 12))
ax.grid(axis="y", alpha=0.25)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, fontsize=8, loc="upper left")
ax.set_title("what the vote buys", fontsize=9, loc="left", color="#555")

axs.plot([s["n"] for s in SPLIT], [s["plurality_accuracy"] * 100 for s in SPLIT],
         marker="o", ms=4.5, lw=2, color="#b23", label=f"the vote, T={T_BEST}")
if any(s["page_06_greedy"] is not None for s in SPLIT):
    have = [s for s in SPLIT if s["page_06_greedy"] is not None]
    axs.plot([s["n"] for s in have], [s["page_06_greedy"] * 100 for s in have],
             marker="s", ms=4, lw=1.4, ls=":", color="#9aa0ab", label="page 6, greedy")
axs.set_xscale("log", base=2)
axs.set_xticks(SPLIT_N)
axs.set_xticklabels([f"{n}\n{B_TOTAL // n}" for n in SPLIT_N], fontsize=8)
axs.set_xlabel(f"chains, and tokens each (total held at {B_TOTAL})")
axs.set_ylim(0, min(100, max(p["coverage"] for p in hot) * 100 + 12))
axs.grid(axis="y", alpha=0.25)
axs.spines[["top", "right"]].set_visible(False)
axs.legend(frameon=False, fontsize=8, loc="upper right")
axs.set_title("what it costs at a fixed budget", fontsize=9, loc="left", color="#555")

fig.suptitle(f"{SWEPT[0].split('/')[-1]}, {DATASET}, n={len(items)}", fontsize=10, x=0.02,
             ha="left", color="#333")
fig.tight_layout()
fig.savefig(OUT_DIR / "figure_2_vote.png", dpi=160)
plt.show()

## 14. The number the page hands to page 8

Page 7's third beat is one measured sentence and no figure: the share of problems where the model
produced the right answer and the vote threw it away. It is `coverage` minus the vote, computed
on the pile page 8 then reads.

In [ ]:
POOL_ROWS = [tally(PRIMARY_SAMPLES[POOL_CONFIG][p["unique_id"]][:POOL_N])
             for p in items if p["unique_id"] in PRIMARY_SAMPLES[POOL_CONFIG]]
A_POOL = aggregate(POOL_ROWS)

HANDOFF = dict(config_id=POOL_CONFIG, n_chains=POOL_N, temperature=T_BEST,
               chain_cap=CHAIN_CAP, n_items=A_POOL["n_items"],
               single_chain_accuracy=A_POOL["single_chain_accuracy"],
               plurality_accuracy=A_POOL["plurality_accuracy"],
               coverage=A_POOL["coverage"], thrown_away=A_POOL["thrown_away"],
               tie_rate=A_POOL["tie_rate"], gain_pts=GAIN_PTS, gain_p=p_value,
               gain_fixed=fixed, gain_broken=broken)

print(f"{POOL_N} chains at T={T_BEST}, {A_POOL['n_items']} problems")
print(f"  one chain        {A_POOL['single_chain_accuracy']:>6.0%}")
print(f"  the vote         {A_POOL['plurality_accuracy']:>6.0%}   ({GAIN_PTS:+.0f} points, "
      f"p={p_value:.4f})")
print(f"  in the pile      {A_POOL['coverage']:>6.0%}")
print(f"\n-> on {A_POOL['thrown_away']:.0%} of problems the model produced the right answer and "
      f"the vote threw it away")
print(f"   ({A_POOL['tie_rate']:.0%} of votes were ties, which count as thrown away)")
if A_POOL["thrown_away"] < 0.05:
    print("\n!! Under five points of gap. Page 8's whole subject is what lives inside this gap,")
    print("   and at this size its ladder has nowhere to climb. Raise POOL_N.")

## 15. Independent verification

Everything above was computed from objects held in memory. This cell re-reads `traces.jsonl` from
disk and recomputes the headline numbers from the raw generations alone, using nothing built
earlier. If the two disagree, the figures are wrong and the notebook says so rather than shipping
them.

In [ ]:
raw = [r for r in read_traces(TRACES_PATH)
       if r["run_id"] == RUN_ID and r["model_id"] == SWEPT[0]
       and r["config_id"] == POOL_CONFIG]
by_item = {}
for r in raw:
    by_item.setdefault(r["problem_id"], []).append(r)

n_cov = n_vote = n_one = 0
for pid, rs in by_item.items():
    rs = sorted(rs, key=lambda r: r["sample_index"])[:POOL_N]
    gt = rs[0]["ground_truth"]
    n_cov += any(r["correct"] for r in rs)
    n_one += bool(rs[0]["correct"])
    labs, _ = cluster([vote_key(r["trace"]) for r in rs])
    c = Counter([l for l in labs if l is not NO_ANSWER]).most_common()
    top = c[0][1] if c else 0
    winners = [a for a, k in c if k == top]
    n_vote += bool(len(winners) == 1 and grade_answer(winners[0], gt))

N = len(by_item)
print(f"recomputed from traces.jsonl, independent of everything above ({N} items):")
print(f"  one chain     {n_one / N:>6.1%}")
print(f"  the vote      {n_vote / N:>6.1%}")
print(f"  coverage      {n_cov / N:>6.1%}")
print(f"  thrown away   {(n_cov - n_vote) / N:>+6.1%}")
assert abs(n_vote / N - A_POOL["plurality_accuracy"]) < 1e-9, \
    "the vote disagrees with the trace file"
assert abs(n_cov / N - A_POOL["coverage"]) < 1e-9, "coverage disagrees with the trace file"
print("\nOK   the figures above match the raw generations exactly")

## 16. Artifacts

`derived.json` is the contract with the page: the page reads it and nothing else, and **no
inference runs at demonstration time.** Its top-level `claims` object is read before any number.

**Which traces get bundled.** Figure 1's specimen in full, because the page lets the reader open
a bar and read the chains that landed there, plus the ranked tallies for a handful of other
problems. Everything else stays in `traces.jsonl`, which ships as a download and is what page 8
consumes.

In [ ]:
N_BUNDLED_PROBLEMS = 6


def bundle_problem(pid):
    rs = PRIMARY_SAMPLES[POOL_CONFIG][pid][:POOL_N]
    row = tally(rs)
    gt = rs[0]["ground_truth"]
    return {
        "problem_id": pid, "level": rs[0]["level"], "subject": rs[0]["subject"],
        "question": rs[0]["question"], "ground_truth": gt,
        "config_id": POOL_CONFIG, "temperature": rs[0]["temperature"],
        "chain_cap": rs[0]["chain_budget"],
        "ranked": [{"answer": a, "votes": c, "correct": bool(grade_answer(a, gt))}
                   for a, c in row["ranked"]],
        "distinct_answers": row["distinct_answers"], "answer_entropy": row["answer_entropy"],
        "plurality_answer": row["plurality_answer"],
        "plurality_correct": row["plurality_correct"],
        "tie": row["tie"], "any_correct": row["any_correct"],
        "no_answer_rate": row["no_answer_rate"],
        # No `tokens` here, deliberately. Pages 5 and 6 reveal a chain token by token
        # and need the per-token strings; this page opens a bar and shows prose, so the
        # array would be ten times the trace it duplicates and would put megabytes of
        # per-token quoting into the web bundle. `traces.jsonl` keeps them.
        "chains": [{k: r[k] for k in ("sample_index", "batch_seed", "group_index", "trace",
                                      "cut_index", "answer_start", "truncated",
                                      "forced", "extracted_answer", "vote_key", "correct",
                                      "own_tokens", "tokens_generated")} for r in rs],
    }


ranked_by_tail = sorted(FIG1_ROWS, key=lambda p: -FIG1_ROWS[p]["distinct_answers"])
bundled_ids = [SPECIMEN_ID] + [p for p in ranked_by_tail
                               if p != SPECIMEN_ID][:N_BUNDLED_PROBLEMS - 1]
print(f"bundling {len(bundled_ids)} problems at {POOL_CONFIG}:")
for pid in bundled_ids:
    r = FIG1_ROWS[pid]
    print(f"  {pid:<34} {r['distinct_answers']:>2} distinct  vote "
          f"{'right' if r['plurality_correct'] else 'wrong':<5}"
          f"{'   <- figure 1 specimen' if pid == SPECIMEN_ID else ''}")

In [ ]:
manifest = {
    "run_id": RUN_ID,
    "notebook": "page_07_frequency_voting.ipynb",
    "notebook_sha256": harness.file_sha256("page_07_frequency_voting.ipynb"),
    "harness": harness.fingerprint(),
    "page": 7, "run_mode": RUN_MODE,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "notebook_git_sha": harness.git_sha(),
    "primary_model": SWEPT[0],
    "models": [model_meta.get(m, {"id": m}) for m in SWEPT],
    "replayed": RUN_MODE == "REPLAY",
    "decoding": {"strategy": "sampled above T=0, greedy at T=0", "top_p": TOP_P,
                 "temperatures": TEMPERATURES, "pile_temperature": T_BEST,
                 "seed_rule": "sha256(SEED, problem_id, config_id, group_index), one per group",
                 "group_size": GROUP,
                 "batching": "the chains of one problem are one generate call with "
                             "num_return_sequences; temperature 0 is generated unbatched",
                 "forced_answer_decoding": "greedy at every temperature"},
    "pile": {"chain_cap": CHAIN_CAP, "n_chains": POOL_N, "config_id": POOL_CONFIG,
             "n_axis": N_AXIS, "n_resamples": N_RESAMPLES,
             "axis_rule": "a point at N is the mean over random N-subsets of the same pile"},
    "split": {"budget_total": B_TOTAL, "n_grid": SPLIT_N,
              "chain_budgets": {str(n): B_TOTAL // n for n in SPLIT_N},
              "answer_allowance": ANSWER_ALLOWANCE,
              "note": "matched in thinking tokens; the forced answer is overhead on top"},
    "forcing": {"stem": FORCE_STEM, "answer_allowance": ANSWER_ALLOWANCE},
    "items": {"source": "MATH-500 (ch03 copy)" if DATASET.startswith("math") else "GSM8K test",
              "dataset": DATASET, "n": len(items), "item_set_sha256": ITEM_SET_HASH,
              "seed": SEED, "same_as_page_6": SAME_ITEMS_AS_PAGE_6},
    "vote": {"tally_key": "reasoning_from_scratch.ch03.extract_final_candidate"
                          " (fallback='number_only')",
             "equivalence": "reasoning_from_scratch.ch03.grade_answer, both directions",
             "tie_rule": "a tie returns no answer and grades wrong (ch04 self_consistency_vote)"},
    "grading": {"extract": "reasoning_from_scratch.ch03.extract_final_candidate",
                "grade": "reasoning_from_scratch.ch03.grade_answer"},
    "prompt_template": render_prompt("{question}"),
    "versions": {"torch": torch.__version__,
                 "transformers": __import__("transformers").__version__,
                 "device": str(device) if device is not None else None},
    "cost": {"planned_tokens": TOTAL_TOKENS, "batched_tok_per_s": BATCH_TOK_S,
             "priced_from": "page 6's measured spend per cap, not the caps"},
    "integrity": INTEGRITY,
}

derived = {
    "manifest": manifest,
    # Read before any number.
    "claims": {
        # The sentence the page wants: voting improves on chain of thought. Bare CoT is
        # one greedy chain, not one sampled one, so this is the gate that blocks.
        "vote_beats_bare_cot": VOTE_BEATS_BARE_COT,
        "vote_beats_one_chain": VOTE_BEATS_ONE_CHAIN,
        "coverage_exceeds_vote": A_POOL["thrown_away"] > 0.0,
        "t0_vote_is_a_noop": T0_IDENTICAL in (None, 1.0),
        "diversity_is_measured": len(D_AGG) >= 2,
        "specimen_usable": not SPECIMEN_IS_FALLBACK,
        "errors_are_diffuse": SHAPE["singleton_share"] > 0.5,
        "splitting_a_fixed_budget_pays": SPLIT_PAYS,
        "pool_supports_page_8": A_POOL["thrown_away"] >= 0.05,
    },
    "procedure": (
        f"Sample N chains of one problem at temperature T with top-p {TOP_P}, each capped at "
        f"{CHAIN_CAP} tokens and forced to commit if it hits the cap; grade with MATH-500's "
        f"reference answer; keep the answer the most chains agree on, where agreement is ch03 "
        f"equivalence and a tie is not a vote. The N axis is drawn from one pile of {POOL_N} "
        f"chains over {len(items)} problems."),
    "figure_1": {
        "config_id": POOL_CONFIG, "n_chains": POOL_N, "temperature": T_BEST,
        "chain_cap": CHAIN_CAP,
        "specimen_id": SPECIMEN_ID, "specimen_is_fallback": SPECIMEN_IS_FALLBACK,
        "n_candidates": len(candidates), "shape": SHAPE,
        "problems": [bundle_problem(pid) for pid in bundled_ids],
    },
    "figure_2": {
        "pile": [{"temperature": t, "points": pts} for t, pts in PILE.items()],
        "split": SPLIT,
        "split_prediction_from_page_6": SPLIT_PREDICTION,
        "splitting_pays": SPLIT_PAYS,
        "best_split_n": peak["n"],
        # `n_items` here is the diversity family's smaller item set, not the pile's.
        "diversity": [dict(temperature=t, n_chains=DIVERSITY_N, **a)
                      for t, a in sorted(D_AGG.items())],
        # Two contrasts, and the page quotes the second. `vs_one_sample` is the
        # self-consistency literature's baseline; `vs_bare_cot` is pages 5 and 6's.
        "gain": dict(vs_one_sample=dict(pts=GAIN_PTS, p=p_value, fixed=fixed,
                                        broken=broken,
                                        baseline=A_POOL["single_chain_accuracy"]),
                     vs_bare_cot=BARE,
                     vote=A_POOL["plurality_accuracy"], n_chains=POOL_N),
    },
    "handoff": HANDOFF,
    "pool": {"config_id": POOL_CONFIG, "n_chains": POOL_N, "chain_cap": CHAIN_CAP,
             "temperature": T_BEST,
             "note": "this page's candidate pile; page 8 re-runs the same protocol on its own chains"},
}

DERIVED_PATH.write_text(json.dumps(derived, indent=2, ensure_ascii=False), encoding="utf-8")
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")

print(f"derived.json   {harness.kb(DERIVED_PATH):>10,.1f} KB   <- bundled with the page")
print(f"manifest.json  {harness.kb(MANIFEST_PATH):>10,.1f} KB   <- bundled with the page")
print(f"traces.jsonl   {harness.kb(TRACES_PATH):>10,.1f} KB   <- download, and page 8's input")
if harness.kb(DERIVED_PATH) > 1024:
    print("\n^ derived.json is over 1 MB. Lower N_BUNDLED_PROBLEMS.")

blocking = ["vote_beats_bare_cot", "coverage_exceeds_vote", "specimen_usable"]
failed = [k for k in blocking if not derived["claims"][k]]
print(f"\n{'TRANSLATABLE' if not failed else 'NOT TRANSLATABLE: ' + ', '.join(failed)}")
for k, v in derived["claims"].items():
    print(f"   {'ok ' if v else 'NO '} {k}")
print("\n`splitting_a_fixed_budget_pays` is NOT blocking and is not expected to be true on this")
print("band. It changes what beat 2's second panel says, not whether the page ships: false")
print("means 'voting buys accuracy, and the budget has to come from somewhere'.")

## 17. Notes for re-derivation

- **A group is the unit of reproduction.** One seed per group of `GROUP` chains, recorded as
  `batch_seed` with `group_index`, so a group regenerates exactly and a single chain regenerates
  only as part of its group. Different hardware, a different group size or a different
  transformers version will produce different chains; the shape and the ranking should survive,
  the individual traces will not.
- **Batching is why this is a two-hour run.** It also changes floating-point reduction order
  against an unbatched run, which is why the temperature-0 line is generated one chain at a time:
  that line has to be comparable to page 6's greedy sweep.
- The vote's equivalence relation is the grader. That is deliberate, and it means the histogram's
  tail cannot be an artifact of notation.
- Ties count as wrong, following ch04. The tie rate is printed beside every point, so a reader
  can see how much of the vote's shortfall is ties rather than wrong consensus.
- **The two axes answer different questions and must not be plotted on one x-axis.** The pile
  holds chain length fixed and varies how many candidates the vote sees. The split holds the
  total budget fixed and varies both. A single curve over N would confound them.
- The split axis is matched in **thinking** tokens; the forced answer is charged on top and grows
  with N. Section 10 prints the real spend.
- Changing the model, band, cap, pile size, top-p or item count changes `RUN_ID`; old traces stay
  in the file as a record and are never replayed.

What the page may claim is [`docs/page_7_refactor.md`](../docs/page_7_refactor.md), checked at
translation ([`docs/page_6_refactor.md`](../docs/page_6_refactor.md) section 6).